# 📊 Prototipo: Explorador de Riesgo de Cancelaciones Hoteleras

Este prototipo interactivo permite explorar los factores que influyen en la cancelación de reservas hoteleras.

🎯 Objetivo:
Identificar patrones asociados al riesgo de cancelación y sugerir acciones para reducir pérdidas y mejorar la ocupación.

💡 Basado en:
- Análisis exploratorio (EDA)
- Definición del problema de negocio
- Variables clave identificadas

In [13]:
# ============================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

sns.set_style("whitegrid")

In [14]:
# ============================================================
# 2. CARGA DE DATOS
# ============================================================

df = pd.read_csv("../data/raw/01-hotel_bookings.csv")

print("Dimensiones del dataset:", df.shape)
df.head()

Dimensiones del dataset: (119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,01-07-15
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,01-07-15
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,02-07-15
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,02-07-15
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,03-07-15


In [15]:
# ============================================================
# 3. LIMPIEZA BÁSICA
# ============================================================

# Eliminar columna con demasiados nulos
df = df.drop(columns=['company'])

# Eliminar outliers extremos de ADR
df = df[df['adr'] < 500]

# Rellenar nulos simples
df['children'] = df['children'].fillna(0)
df['country'] = df['country'].fillna('Unknown')

## 🎛️ Panel de Exploración

Seleccione filtros para analizar el riesgo de cancelación en diferentes escenarios.

In [16]:
# ============================================================
# 4. WIDGETS INTERACTIVOS
# ============================================================

hotel_widget = widgets.Dropdown(
    options=['Todos'] + sorted(df['hotel'].unique().tolist()),
    description='Hotel:'
)

deposit_widget = widgets.Dropdown(
    options=['Todos'] + sorted(df['deposit_type'].unique().tolist()),
    description='Depósito:'
)

market_widget = widgets.Dropdown(
    options=['Todos'] + sorted(df['market_segment'].unique().tolist()),
    description='Segmento:'
)

month_widget = widgets.Dropdown(
    options=['Todos'] + sorted(df['arrival_date_month'].unique().tolist()),
    description='Mes:'
)

lead_slider = widgets.IntSlider(
    value=100,
    min=0,
    max=500,
    step=10,
    description='Lead Time ≤'
)

adr_slider = widgets.FloatSlider(
    value=150,
    min=0,
    max=500,
    step=10,
    description='ADR ≤'
)

In [ ]:
# ============================================================
# 5. FUNCIÓN PRINCIPAL DEL PROTOTIPO
# ============================================================

def analizar_cancelaciones(hotel, deposit, market, month, lead_time, adr):

    data = df.copy()

    if hotel != 'Todos':
        data = data[data['hotel'] == hotel]

    if deposit != 'Todos':
        data = data[data['deposit_type'] == deposit]

    if market != 'Todos':
        data = data[data['market_segment'] == market]

    if month != 'Todos':
        data = data[data['arrival_date_month'] == month]

    data = data[data['lead_time'] <= lead_time]
    data = data[data['adr'] <= adr]

    if len(data) == 0:
        print("⚠️ No hay datos con estos filtros")
        return

    # KPI principal
    tasa = data['is_canceled'].mean()

    print("="*50)
    print(f"📌 Tasa de cancelación: {tasa:.2%}")
    print(f"📊 Registros analizados: {len(data)}")
    print("="*50)

    # Histograma
    plt.figure(figsize=(6,4))
    sns.histplot(data['lead_time'], bins=30)
    plt.title("Distribución de Lead Time")
    plt.show()

    # Cancelación por tipo de cliente
    plt.figure(figsize=(6,4))
    data.groupby('customer_type')['is_canceled'].mean().plot(kind='bar', color='coral')
    plt.title("Cancelación por Tipo de Cliente")
    plt.ylabel("Tasa de cancelación")
    plt.show()

    # ==========================
    # RECOMENDACIONES
    # ==========================

    print("\n💡 Recomendaciones:")

    if tasa > 0.4:
        print("- Alta probabilidad de cancelación")
        print("- Aplicar políticas de depósito más estrictas")
        print("- Enviar recordatorios previos a la llegada")

    if lead_time > 200:
        print("- Reservas con mucha anticipación tienden a cancelarse")

    if adr > 150:
        print("- Precios altos pueden aumentar cancelaciones")

    if deposit == 'No Deposit':
        print("- Considerar depósitos para reducir riesgo")

    if len(data[data['total_of_special_requests'] > 0]) > 0:
        print("- Clientes con solicitudes especiales tienden a cancelar menos")

In [17]:
# ============================================================
# 6. EJECUCIÓN INTERACTIVA
# ============================================================

widgets.interact(
    analizar_cancelaciones,
    hotel=hotel_widget,
    deposit=deposit_widget,
    market=market_widget,
    month=month_widget,
    lead_time=lead_slider,
    adr=adr_slider
)

interactive(children=(Dropdown(description='Hotel:', options=('Todos', 'City Hotel', 'Resort Hotel'), value='T…

<function __main__.analizar_cancelaciones(hotel, deposit, market, month, lead_time, adr)>

In [18]:
# ============================================================
# 7. EXPLORADOR DE VARIABLES
# ============================================================

col_num = df.select_dtypes(include=np.number).columns.tolist()

selector_variable = widgets.Dropdown(
    options=col_num,
    description='Variable:'
)

def explorar_variable(variable):
    plt.figure(figsize=(6,4))
    sns.histplot(df[variable].dropna(), bins=30)
    plt.title(f"Distribución de {variable}")
    plt.show()

widgets.interact(explorar_variable, variable=selector_variable)

interactive(children=(Dropdown(description='Variable:', options=('is_canceled', 'lead_time', 'arrival_date_yea…

<function __main__.explorar_variable(variable)>